# 📊 Assignment 2: Data Classification

## 📌 Overview
This assignment applies multiple classification models on the MAGIC Gamma Telescope dataset. The objective is to preprocess the data, handle class imbalance, train different models, tune parameters, and evaluate performance using standard metrics.

## 📌 Steps Performed

### 1. Data Loading & Exploration
The dataset was loaded and inspected using basic functions such as head(), info(), and describe(). No missing values or duplicates were found.

### 2. Data Balancing
The dataset was imbalanced (gamma class larger than hadron). To fix this, we downsampled the gamma class to match the hadron class size.

### 3. Data Preprocessing
- Column names were assigned
- Labels were converted from categorical (g/h) to numerical (1/0)
- Data was shuffled randomly

### 4. Data Splitting
The dataset was split into:
- 70% Training data
- 30% Testing data

### 5. Models Used
- Decision Tree (no tuning required)
- Naive Bayes (no tuning required)
- Random Forest (tuned using n_estimators)
- AdaBoost (tuned using n_estimators)

### 6. Parameter Tuning
Cross-validation (GridSearchCV) was used to select the best parameters for Random Forest and AdaBoost.

### 7. Evaluation Metrics
The models were evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

### 8. Results & Detailed Comparison

#### Decision Tree
- Accuracy: 0.7899
- Precision: 0.7759
- Recall: 0.8062
- F1-score: 0.7908
- The model shows balanced performance but has some misclassification between classes.

#### Naive Bayes
- Accuracy: 0.6409
- Precision: 0.5888
- Recall: 0.8978
- F1-score: 0.7112
- This model has very high recall but low precision, meaning it detects most positive cases but produces many false positives.

#### Random Forest
- Accuracy: 0.8520
- Precision: 0.8272
- Recall: 0.8841
- F1-score: 0.8547
- This model achieved the best performance overall, with high accuracy and a good balance between precision and recall.

#### AdaBoost
- Accuracy: 0.8004
- Precision: 0.7896
- Recall: 0.8107
- F1-score: 0.8000
- AdaBoost performed better than Decision Tree and Naive Bayes, but slightly lower than Random Forest.

### 9. Final Comparison
- Random Forest achieved the highest accuracy and best overall performance.
- Naive Bayes had the highest recall but lowest precision.
- AdaBoost provided balanced results but did not outperform Random Forest.
- Decision Tree performed reasonably well but may suffer from overfitting.

### 10. Conclusion
Random Forest is the best model for this dataset as it provides the highest accuracy and a strong balance between precision and recall. Ensemble methods (Random Forest and AdaBoost) outperform individual models, showing the effectiveness of combining multiple learners.

---

In [3]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [4]:
# Load dataset
data = pd.read_csv("magic04.data")

In [5]:
data.head()

,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.011,-8.2027,40.092,81.8828,g
0,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.261,g
1,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.788,g
2,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.737,g
3,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.462,g
4,51.6240,21.1502,2.9085,0.2420,0.1340,50.8761,43.1887,9.8145,3.6130,238.098,g


In [6]:
data.columns = [
    'fLength','fWidth','fSize','fConc','fConc1',
    'fAsym','fM3Long','fM3Trans','fAlpha','fDist','class'
]

In [7]:
data.sample(5)

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
17225,147.5111,49.7622,3.2971,0.2135,0.1136,-144.8928,-91.3556,-39.1199,33.4438,254.8939,h
13972,48.6294,15.5835,3.0432,0.3324,0.1861,35.9226,50.1272,-4.3105,71.1767,140.7900,h
17866,34.5557,15.2649,2.6753,0.2957,0.1510,45.8224,22.8035,-8.1748,52.9638,221.0130,h
10883,42.2087,17.1098,2.7649,0.2801,0.1555,-19.0349,27.2251,12.3185,50.5210,35.1638,g
5847,105.1250,17.8888,2.7177,0.4559,0.3209,-67.2858,-65.8081,-15.5054,8.6480,332.1750,g


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19019 entries, 0 to 19018
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   fLength   19019 non-null  float64
 1   fWidth    19019 non-null  float64
 2   fSize     19019 non-null  float64
 3   fConc     19019 non-null  float64
 4   fConc1    19019 non-null  float64
 5   fAsym     19019 non-null  float64
 6   fM3Long   19019 non-null  float64
 7   fM3Trans  19019 non-null  float64
 8   fAlpha    19019 non-null  float64
 9   fDist     19019 non-null  float64
 10  class     19019 non-null  object 
dtypes: float64(10), object(1)
memory usage: 1.6+ MB


In [9]:
data.describe()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist
count,19019.000000,19019.000000,19019.000000,19019.000000,19019.000000,19019.000000,19019.000000,19019.000000,19019.000000,19019.000000
mean,53.251440,22.181291,2.825026,0.380326,0.214658,-4.333429,10.544942,0.250170,27.645052,193.823912
std,42.365598,18.346484,0.472609,0.182818,0.110514,59.207163,51.001391,20.827896,26.104151,74.729344
min,4.283500,0.000000,1.941300,0.013100,0.000300,-457.916100,-331.780000,-205.894700,0.000000,1.282600
25%,24.336000,11.863700,2.477100,0.235800,0.128450,-20.588300,-12.845050,-10.849750,5.546950,142.499000
50%,37.149000,17.140600,2.739600,0.354100,0.196500,4.011900,15.309400,0.689800,17.677000,191.856900
75%,70.126850,24.739950,3.101600,0.503700,0.285250,24.060350,35.844100,10.947050,45.884100,240.564550
max,334.177000,256.382000,5.323300,0.893000,0.675200,575.240700,238.321000,179.851000,90.000000,495.561000


In [10]:
print(data.isnull().sum())

fLength     0
fWidth      0
fSize       0
fConc       0
fConc1      0
fAsym       0
fM3Long     0
fM3Trans    0
fAlpha      0
fDist       0
class       0
dtype: int64


In [11]:
data.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
19014    False
19015    False
19016    False
19017    False
19018    False
Length: 19019, dtype: bool

In [12]:
# Separate classes
gamma = data[data['class'] == 'g']
hadron = data[data['class'] == 'h']

# Downsample gamma to match hadron size
gamma_balanced = gamma.sample(len(hadron), random_state=42)

# Combine again
data_balanced = pd.concat([gamma_balanced, hadron])

In [13]:
# Shuffle the dataset
data_balanced = data_balanced.sample(frac=1, random_state=42)

In [14]:
data_balanced['class'] = data_balanced['class'].map({'g': 1, 'h': 0})

In [15]:
X = data_balanced.drop('class', axis=1)
y = data_balanced['class']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [17]:
def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [18]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

evaluate(y_test, y_pred_dt, "Decision Tree")


Decision Tree
Accuracy: 0.7899327186643409
Precision: 0.7759376522162689
Recall: 0.8061740890688259
F1-score: 0.7907669396872673
Confusion Matrix:
 [[1577  460]
 [ 383 1593]]


In [19]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

evaluate(y_test, y_pred_nb, "Naive Bayes")


Naive Bayes
Accuracy: 0.6409170196860204
Precision: 0.5887819449054099
Recall: 0.8977732793522267
F1-score: 0.7111645620364803
Confusion Matrix:
 [[ 798 1239]
 [ 202 1774]]


In [20]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [10, 50, 100]
}

grid_rf = GridSearchCV(rf, param_grid, cv=5)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

evaluate(y_test, y_pred_rf, "Random Forest")


Random Forest
Accuracy: 0.8519810615499627
Precision: 0.8271780303030303
Recall: 0.8841093117408907
F1-score: 0.8546966731898239
Confusion Matrix:
 [[1672  365]
 [ 229 1747]]


In [21]:
ada = AdaBoostClassifier(random_state=42)

param_grid = {
    'n_estimators': [10, 50, 100]
}

grid_ada = GridSearchCV(ada, param_grid, cv=5)
grid_ada.fit(X_train, y_train)

best_ada = grid_ada.best_estimator_
y_pred_ada = best_ada.predict(X_test)

evaluate(y_test, y_pred_ada, "AdaBoost")


AdaBoost
Accuracy: 0.8003987042113132
Precision: 0.7895515032035485
Recall: 0.8107287449392713
F1-score: 0.8
Confusion Matrix:
 [[1610  427]
 [ 374 1602]]
